# Taller Interactivo de Gromov-Wasserstein con Swiss Roll

**Audiencia**
- Estudiantes que ya han visto nociones basicas de algebra lineal, distancias y aprendizaje no supervisado.

**Prerrequisitos**
- Saber leer tablas de `pandas` y ejecutar notebooks en Google Colab.
- Entender que representa una nube de puntos en 2D o 3D.
- Tener una idea general de que compara una distancia entre distribuciones.

**Objetivo del taller**
- Generar un unico dataset `Swiss Roll`.
- Visualizar su geometria en 3D.
- Construir la geometria que Gromov-Wasserstein va a comparar.
- Obtener una version 2D proyectada del mismo dataset.

**Idea central**
- Gromov-Wasserstein compara relaciones internas entre espacios.
- En este notebook usamos esa idea para pasar de un rollo en 3D a una representacion plana en 2D.


## Ruta del notebook

1. Preparar el entorno y las funciones auxiliares.
2. Generar un unico `Swiss Roll`.
3. Inspeccionar y visualizar el rollo en 3D.
4. Instalar POT y definir el bloque de Gromov-Wasserstein.
5. Construir la referencia 2D y las matrices de distancias.
6. Resolver GW y obtener la proyeccion 2D.
7. Interpretar el resultado final.


## Paso 0 - Preparar el entorno

Primero dejamos listas las herramientas base del notebook: generacion del `Swiss Roll`, visualizaciones y utilidades numericas.


In [ ]:
from __future__ import annotations

from IPython.display import display

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.datasets import make_swiss_roll

pio.renderers.default = "colab"


def min_max_scale(values):
    values = np.asarray(values, dtype=float)
    span = values.max() - values.min()
    if span == 0:
        return np.zeros_like(values)
    return (values - values.min()) / span


def build_swiss_roll_frame(n_samples=300, noise=0.05, random_state=7):
    points, parameter_t = make_swiss_roll(
        n_samples=n_samples,
        noise=noise,
        random_state=random_state,
    )
    swiss_roll_frame = pd.DataFrame(points, columns=["x", "y", "z"])
    swiss_roll_frame["t"] = parameter_t
    return swiss_roll_frame


def plot_swiss_roll_3d(data, title):
    fig = px.scatter_3d(
        data,
        x="x",
        y="y",
        z="z",
        color="t",
        color_continuous_scale="Turbo",
        labels={"t": "Parametro t"},
        opacity=0.80,
        title=title,
    )
    fig.update_traces(marker={"size": 4})
    fig.update_layout(
        template="plotly_white",
        height=700,
        scene={
            "xaxis_title": "X",
            "yaxis_title": "Y",
            "zaxis_title": "Z",
        },
        coloraxis_colorbar={"title": "Parametro t"},
    )
    return fig


def plot_plane_2d(data, title, x_col="u", y_col="v"):
    fig = px.scatter(
        data,
        x=x_col,
        y=y_col,
        color="t",
        color_continuous_scale="Turbo",
        labels={x_col: "U", y_col: "V", "t": "Parametro t"},
        title=title,
    )
    fig.update_traces(marker={"size": 8, "opacity": 0.85})
    fig.update_layout(
        template="plotly_white",
        height=650,
        xaxis_title="U",
        yaxis_title="V",
        coloraxis_colorbar={"title": "Parametro t"},
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig


print("Entorno base listo para trabajar con el Swiss Roll.")


## Paso 1 - Generar el dataset

Trabajaremos con un solo dataset durante todo el flujo. Como Gromov-Wasserstein puede ser costoso, usamos por defecto `300` puntos para que el notebook corra bien en Google Colab.


In [ ]:
N_SAMPLES = 300
NOISE = 0.05
RANDOM_STATE = 7

swiss_roll_df = build_swiss_roll_frame(
    n_samples=N_SAMPLES,
    noise=NOISE,
    random_state=RANDOM_STATE,
)

print(f"Forma del dataset: {swiss_roll_df.shape}")
display(swiss_roll_df.head(10))


## Paso 2 - Leer el dataset antes de proyectarlo

Antes de usar GW conviene mirar dos cosas:
- un resumen numerico del dataset,
- una visualizacion 3D del rollo original.

El color representa el parametro `t`, que recorre el rollo y luego nos ayudara a leer el resultado 2D.


In [ ]:
summary = swiss_roll_df.describe().T.loc[:, ["mean", "std", "min", "max"]]
summary.round(2)


In [ ]:
swiss_roll_fig = plot_swiss_roll_3d(
    swiss_roll_df,
    title="Swiss Roll original en 3D",
)

swiss_roll_fig


## Paso 3 - Instalar POT y preparar el bloque de GW

Colab normalmente ya tiene `numpy`, `pandas`, `scikit-learn` y `plotly`, pero no siempre trae POT. Esta libreria es la que resolvera el acoplamiento de Gromov-Wasserstein.


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "POT"])
print("POT instalado y listo para usar.")


In [ ]:
import ot
from scipy.sparse.csgraph import shortest_path
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import kneighbors_graph


def build_flat_target_strip(source_data, plane_length=3.0):
    u = plane_length * min_max_scale(source_data["t"].to_numpy())
    v = min_max_scale(source_data["y"].to_numpy()) - 0.5
    flat_target = pd.DataFrame({"u": u, "v": v, "t": source_data["t"].to_numpy()})
    return flat_target


def normalize_distance_matrix(matrix):
    matrix = np.asarray(matrix, dtype=float)
    max_value = matrix.max()
    if max_value == 0:
        return matrix
    return matrix / max_value


def compute_source_geodesic_matrix(source_points, neighbor_candidates=(8, 10, 12, 15, 20)):
    for n_neighbors in neighbor_candidates:
        graph = kneighbors_graph(
            source_points,
            n_neighbors=n_neighbors,
            mode="distance",
            include_self=False,
        )
        graph = graph.maximum(graph.T)
        geodesic_distances = shortest_path(graph, directed=False, unweighted=False)
        if np.isfinite(geodesic_distances).all():
            return normalize_distance_matrix(geodesic_distances), n_neighbors
    raise ValueError("No fue posible construir un grafo conexo para el Swiss Roll.")


def compute_target_distance_matrix(target_points):
    target_distances = pairwise_distances(target_points, metric="euclidean")
    return normalize_distance_matrix(target_distances)


def solve_gw_plan(source_distances, target_distances):
    n_source = source_distances.shape[0]
    n_target = target_distances.shape[0]
    source_weights = np.full(n_source, 1.0 / n_source)
    target_weights = np.full(n_target, 1.0 / n_target)

    coupling, log = ot.gromov.gromov_wasserstein(
        source_distances,
        target_distances,
        source_weights,
        target_weights,
        loss_fun="square_loss",
        max_iter=200,
        log=True,
    )
    return coupling, log


def barycentric_projection(coupling, target_coordinates):
    row_masses = coupling.sum(axis=1, keepdims=True)
    row_masses = np.maximum(row_masses, 1e-12)
    return (coupling @ target_coordinates) / row_masses


def build_unrolled_frame(source_data, transported_coordinates):
    unrolled = source_data.copy().reset_index(drop=True)
    unrolled["u"] = transported_coordinates[:, 0]
    unrolled["v"] = transported_coordinates[:, 1]
    return unrolled


print("Funciones de Gromov-Wasserstein listas.")


## Paso 4 - Construir la referencia 2D

La proyeccion final no sale de una proyeccion euclidea directa desde 3D. Primero definimos una referencia 2D con la que GW va a comparar la geometria del rollo.

Esta referencia es interpretable:
- `u` sigue el avance sobre el rollo segun `t`,
- `v` usa la altura normalizada.

Conviene verla por separado antes de resolver el acoplamiento.


In [ ]:
PLANE_LENGTH = 3.0

flat_target_df = build_flat_target_strip(
    swiss_roll_df,
    plane_length=PLANE_LENGTH,
)

display(flat_target_df.head(10))


In [ ]:
flat_target_fig = plot_plane_2d(
    flat_target_df,
    title="Referencia 2D para la comparacion geometrica",
)

flat_target_fig


## Paso 5 - Construir la geometria que GW va a comparar

Aqui aparece la parte importante del metodo:
- para el `Swiss Roll` aproximamos distancias geodesicas con un grafo `k`-NN y caminos minimos,
- para la referencia 2D usamos distancias euclideas en el plano.

GW trabajara sobre estas matrices de distancias, no sobre las coordenadas crudas.


In [ ]:
GRAPH_NEIGHBORS = (8, 10, 12, 15, 20)

source_points = swiss_roll_df[["x", "y", "z"]].to_numpy()
target_points = flat_target_df[["u", "v"]].to_numpy()

source_geodesic_matrix, used_neighbors = compute_source_geodesic_matrix(
    source_points,
    neighbor_candidates=GRAPH_NEIGHBORS,
)
target_distance_matrix = compute_target_distance_matrix(target_points)

print(f"Vecinos usados para conectar el grafo: {used_neighbors}")
print(f"Forma de la matriz del Swiss Roll: {source_geodesic_matrix.shape}")
print(f"Forma de la matriz 2D: {target_distance_matrix.shape}")


## Paso 6 - Resolver GW y obtener la proyeccion 2D

Ahora si resolvemos el acoplamiento de Gromov-Wasserstein. A partir de ese acoplamiento usamos una proyeccion baricentrica para asignar coordenadas planas al mismo conjunto de puntos del `Swiss Roll`.


In [ ]:
gw_coupling, gw_log = solve_gw_plan(
    source_geodesic_matrix,
    target_distance_matrix,
)
transported_coordinates = barycentric_projection(gw_coupling, target_points)
unrolled_df = build_unrolled_frame(swiss_roll_df, transported_coordinates)

gw_distance = gw_log.get("gw_dist", np.nan)
print(f"Forma de la matriz de acoplamiento: {gw_coupling.shape}")
print(f"Distancia GW reportada: {gw_distance:.6f}")
display(unrolled_df[["u", "v", "t"]].head(10))


In [ ]:
unrolled_fig = plot_plane_2d(
    unrolled_df,
    title="Swiss Roll proyectado a 2D con Gromov-Wasserstein",
)

unrolled_fig


## Interpretacion final

La historia completa del notebook es esta:
- empezamos con un rollo en 3D,
- construimos una referencia 2D interpretable,
- comparamos ambos espacios por sus distancias internas,
- y obtenemos una version proyectada del mismo dataset.

La intuicion importante es que GW no se basa solo en coordenadas absolutas. Lo que intenta preservar es la organizacion geometrica interna del espacio.

**Notas practicas**
- Si quieres mas detalle visual, prueba `N_SAMPLES = 350` o `400`.
- Si subes demasiado ese valor, el calculo de GW puede volverse lento en Colab.
